# 05 · SharePoint and the identity boundary

## Goal

Attach the supplier contracts SharePoint library as a knowledge source, and
prove — not assert — that the agent sees what the asking user can see. Run
the same question as two different users and get two different answers.
That pair of runs is the actual deliverable of this notebook.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("SHAREPOINT_SITE_URL")
print("SharePoint site configured:", settings.get("SHAREPOINT_SITE_URL"))


Requires two test users already provisioned in your tenant: one in the group with access to the confidential pricing annex (`procurement_lead`), one not (`restricted_user`). This notebook doesn't create them — that's a tenant-admin action outside `pac`'s scope.


## Concept

This is the real lesson of the Knowledge track, not a footnote to it.
SharePoint knowledge sources are security-trimmed: the agent queries with
the asking user's identity via `Authenticate with Microsoft`, and SharePoint's
own permissions decide what comes back — the agent has no separate
all-access view to accidentally leak from. Tenant graph grounding (semantic
search across the tenant) has its own licence dependency worth checking
before you assume it's available.

The consequence for how you *evaluate* this knowledge source: a single
golden case with one expected answer is wrong by construction, because the
correct answer depends on who's asking. `evals/golden_cases.json` carries
`know-03` and `know-04` as a matched pair — same prompt, different `run_as`
persona, opposite expectations — and `csx.verify.run_suite` needs to run
both identities to actually test this, not just one.


## Build


### Attach the SharePoint source


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

source = {
    "id": "supplier-contracts-sharepoint",
    "type": "sharepoint",
    "displayName": "Supplier contracts (SharePoint)",
    "description": "Signed contracts and confidential pricing annexes for all suppliers. Access follows SharePoint site permissions.",
    "siteUrl": settings.get("SHAREPOINT_SITE_URL"),
    "authenticateWithMicrosoft": True,  # required — this is what makes trimming work
}
(workspace / "knowledge" / "supplier-contracts-sharepoint.yaml").write_text(yaml.dump(source, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


In [ ]:
from csx.checkpoint import checkpoint
checkpoint(
    name="Authenticate with Microsoft consent granted for the knowledge connection",
    probe=lambda: input("Have both test users completed first-run consent on the SharePoint connection? (y/n): ") == "y",
    remediation="Sign in as each test user in the Teams/web chat once, complete the Microsoft sign-in prompt, then re-run.",
)


## Verify

Same harness, same golden set, every notebook.


Two clients, two identities, same prompt — this is the actual test.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

# In practice each identity needs its own delegated token — get_copilot_client
# is called once per persona with a token acquired as that user.
client_restricted = get_copilot_client(settings, delegated=True)   # sign in as restricted_user when prompted
restricted_case = [c for c in load_golden(tags=["sharepoint"]) if c["run_as"] == "restricted_user"]
suite_restricted = run_suite(client_restricted, cases=restricted_case, credit_meter=meter, min_pass_rate=1.0)

client_lead = get_copilot_client(settings, delegated=True)         # sign in as procurement_lead when prompted
lead_case = [c for c in load_golden(tags=["sharepoint"]) if c["run_as"] == "procurement_lead"]
suite_lead = run_suite(client_lead, cases=lead_case, credit_meter=meter, min_pass_rate=1.0)

print("If both suites passed: restricted_user was correctly declined, procurement_lead correctly got the citation.")


In [ ]:
from csx.verify import run_suite, load_golden
core = load_golden(tags=["core"])
suite_core = run_suite(client_lead, cases=core, credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("05", budget=settings.get("COPILOT_CREDIT_BUDGET"),
                   delta_credits=(suite_restricted.total_credits + suite_lead.total_credits + suite_core.total_credits),
                   note="SharePoint attach + two-identity trimming proof")


## Teardown


In [ ]:
print("No teardown — SharePoint source persists. Revoke the two test users' consent only if you're decommissioning them.")
